# Meta-Physical Graph Transformer for IEEE 33-Bus DNR

This notebook solves Distribution Network Reconfiguration (DNR) with a custom **Meta-Physical Graph Transformer (Meta-PhyGT)** trained using **Reptile meta-learning** and a **physics-informed multi-objective loss**.

It expects `dnr_diagnostic.h5` generated by `01_DNR_Data_Generation.ipynb`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# GLOBAL HYPERPARAMETERS — tune here first
# ═══════════════════════════════════════════════════════════════════
inner_lr = 1e-3
meta_lr = 1e-4
inner_steps = 5
epochs = 50
w_bce = 1.0
w_mse = 0.5
w_volt = 2.0

# Practical execution controls
support_size = 5
query_size = 16
validation_batch_size = 64
validation_interval = 5
critic_cases_per_eval = 16
random_seed = 42

# Model dimensions
hidden_dim = 96
num_heads = 4
num_layers = 3
structural_dim = 6
dropout = 0.10

# Files and electrical constants
DIAG_FILE = 'dnr_diagnostic.h5'
V_MIN_PU = 0.90
VN_KV = 12.66
BASE_MVA = 10.0
N_BUS = 33
N_LINES = 37
N_OPEN = 5
DEVICE = 'cuda'  # auto-downgraded below if CUDA is unavailable

## CHAPTER 1: DATA PIPELINE & PyG GRAPH DATASET CONSTRUCTION

This chapter inspects the HDF5 artifact, builds the immutable IEEE 33-bus graph, and converts each scenario into a PyTorch Geometric `Data` object.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Imports and deterministic setup
# ═══════════════════════════════════════════════════════════════════
import ast
import copy
import math
import random
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import networkx as nx

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_batch

import pandapower as pp
import pandapower.networks as pn

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

try:
    display
except NameError:
    def display(obj):
        print(obj)

random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)
if torch.cuda.is_available() and DEVICE == 'cuda':
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 1.1 Inspect HDF5 dynamically with h5py, then load pandas tables
# ═══════════════════════════════════════════════════════════════════
if not Path(DIAG_FILE).exists():
    raise FileNotFoundError(
        f'{DIAG_FILE} not found. Run 01_DNR_Data_Generation.ipynb first.'
    )

print('HDF5 structure from h5py:')
h5_inventory = []
with h5py.File(DIAG_FILE, 'r') as h5:
    def visit(name, obj):
        kind = type(obj).__name__
        shape = getattr(obj, 'shape', '')
        h5_inventory.append((name, kind, shape))
        print(f'  {name:40s} {kind:18s} {shape}')
    h5.visititems(visit)

required_keys = ['scenarios', 'summary']
optional_keys = ['training_matrix_check', 'loss_weights']
with pd.HDFStore(DIAG_FILE, mode='r') as store:
    available_keys = {k.strip('/') for k in store.keys()}
    missing = sorted(set(required_keys) - available_keys)
    if missing:
        raise KeyError(f'Missing required HDF5 keys: {missing}')
    print('\nPandas HDF keys:', sorted(available_keys))

scenarios_df = pd.read_hdf(DIAG_FILE, key='scenarios')
summary_df = pd.read_hdf(DIAG_FILE, key='summary')
training_matrix_check = pd.read_hdf(DIAG_FILE, key='training_matrix_check') if 'training_matrix_check' in available_keys else pd.DataFrame()
loss_weights_df = pd.read_hdf(DIAG_FILE, key='loss_weights') if 'loss_weights' in available_keys else pd.DataFrame()

print(f'Loaded scenarios: {scenarios_df.shape}')
display(summary_df.head())
if not training_matrix_check.empty:
    display(training_matrix_check.head())
if not loss_weights_df.empty:
    display(loss_weights_df.head())

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 1.2 IEEE 33-bus topology: pandapower baseline + standard tie lines
# ═══════════════════════════════════════════════════════════════════
STD_FROM_BUS = np.array([
    1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,
    2,19,20,21,3,23,24,6,26,27,28,29,30,31,32,
    21,9,12,18,25
], dtype=np.int64) - 1
STD_TO_BUS = np.array([
    2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,
    19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,
    8,15,22,33,29
], dtype=np.int64) - 1
STD_R_OHM = np.array([
    0.0922,0.4930,0.3660,0.3811,0.8190,0.1872,1.7114,1.0300,1.0440,
    0.1966,0.3744,1.4680,0.5416,0.5910,0.7463,1.2890,0.7320,
    0.1640,1.5042,0.4095,0.7089,0.4512,0.8980,0.8960,0.2030,0.2842,
    1.0590,0.8042,0.5075,0.9744,0.3105,0.3410,2.0000,2.0000,2.0000,0.5000,0.5000
], dtype=np.float64)
STD_X_OHM = np.array([
    0.0470,0.2511,0.1864,0.1941,0.7070,0.6188,1.2351,0.7400,0.7400,
    0.0650,0.1238,1.1550,0.7129,0.5260,0.5450,1.7210,0.5740,
    0.1565,1.3554,0.4784,0.9373,0.3083,0.7091,0.7011,0.1034,0.1447,
    0.9337,0.7006,0.2585,0.9630,0.3619,0.5302,2.0000,2.0000,2.0000,0.5000,0.5000
], dtype=np.float64)

def load_pandapower_case33():
    """Try pandapower's IEEE 33-bus case; fall back to a constructed network."""
    if hasattr(pn, 'case33'):
        net = pn.case33()
        source = 'pn.case33()'
    elif hasattr(pn, 'case33bw'):
        net = pn.case33bw()
        source = 'pn.case33bw()'
    else:
        net = pp.create_empty_network(sn_mva=BASE_MVA)
        for _ in range(N_BUS):
            pp.create_bus(net, vn_kv=VN_KV)
        pp.create_ext_grid(net, bus=0, vm_pu=1.0)
        for fb, tb, r, x in zip(STD_FROM_BUS, STD_TO_BUS, STD_R_OHM, STD_X_OHM):
            pp.create_line_from_parameters(
                net, int(fb), int(tb), length_km=1.0,
                r_ohm_per_km=float(r), x_ohm_per_km=float(x),
                c_nf_per_km=0.0, max_i_ka=1.0
            )
        source = 'constructed standard IEEE-33 network'
    return net, source

base_net, pp_source = load_pandapower_case33()
print(f'Pandapower baseline source: {pp_source}')

# The DNR action space is the full 37-line graph. If pandapower provides fewer
# lines, use the standard 32 section + 5 tie graph for immutable connectivity.
if len(base_net.line) >= N_LINES:
    from_bus = base_net.line.from_bus.to_numpy(dtype=np.int64)[:N_LINES]
    to_bus = base_net.line.to_bus.to_numpy(dtype=np.int64)[:N_LINES]
    r_ohm = base_net.line.r_ohm_per_km.to_numpy(dtype=np.float64)[:N_LINES]
    x_ohm = base_net.line.x_ohm_per_km.to_numpy(dtype=np.float64)[:N_LINES]
else:
    from_bus, to_bus, r_ohm, x_ohm = STD_FROM_BUS, STD_TO_BUS, STD_R_OHM, STD_X_OHM

edge_pairs_np = np.stack([from_bus, to_bus], axis=0).astype(np.int64)
edge_index = torch.as_tensor(edge_pairs_np, dtype=torch.long)
edge_r = torch.as_tensor(r_ohm, dtype=torch.float32)
edge_x = torch.as_tensor(x_ohm, dtype=torch.float32)
line_ids = np.arange(1, N_LINES + 1)
tie_line_mask = line_ids > 32

assert edge_index.shape == (2, 37), edge_index.shape
print('edge_index shape:', tuple(edge_index.shape))
print('Tie-switch line numbers:', line_ids[tie_line_mask].tolist())

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 1.3 Scenario feature extraction and PyG Dataset
# ═══════════════════════════════════════════════════════════════════
p_cols = [f'p_bus_{b}' for b in range(1, N_BUS)]
q_cols = [f'q_bus_{b}' for b in range(1, N_BUS)]
missing_cols = [c for c in p_cols + q_cols if c not in scenarios_df.columns]
if missing_cols:
    raise KeyError(f'Missing load columns in HDF5 scenarios: {missing_cols[:5]}')

season_vocab = ['winter', 'summer', 'spring', 'autumn', 'steered', 'stress']
season_to_idx = {s: i for i, s in enumerate(season_vocab)}

def infer_task(row):
    season = str(row.get('season', 'unknown'))
    if season == 'steered':
        return 'B_topology_steered'
    if season == 'stress':
        return 'C_stress_sampler'
    return 'A_realistic'

def parse_open_lines(topo_value):
    if isinstance(topo_value, (list, tuple, np.ndarray)):
        return np.array(topo_value, dtype=np.int64)
    return np.array(ast.literal_eval(str(topo_value)), dtype=np.int64)

def scenario_load_vectors(row):
    p = np.zeros(N_BUS, dtype=np.float32)
    q = np.zeros(N_BUS, dtype=np.float32)
    p[1:] = row[p_cols].to_numpy(dtype=np.float32)
    q[1:] = row[q_cols].to_numpy(dtype=np.float32)
    return p, q

p_all = scenarios_df[p_cols].to_numpy(dtype=np.float32)
q_all = scenarios_df[q_cols].to_numpy(dtype=np.float32)
p_scale = max(float(np.nanmax(p_all)), 1e-6)
q_scale = max(float(np.nanmax(q_all)), 1e-6)
loss_mu = float(scenarios_df['loss_mw'].mean())
loss_sigma = float(scenarios_df['loss_mw'].std() + 1e-6)
vmin_mu = float(scenarios_df['v_min_pu'].mean())
vmin_sigma = float(scenarios_df['v_min_pu'].std() + 1e-6)

bus_id_norm = np.arange(N_BUS, dtype=np.float32) / (N_BUS - 1)
slack_flag = (np.arange(N_BUS) == 0).astype(np.float32)

def build_node_features(row):
    p, q = scenario_load_vectors(row)
    hour = float(row.get('hour', -1))
    hour_norm = 0.0 if hour < 0 else hour / 23.0
    non_time_flag = 1.0 if hour < 0 else 0.0
    ev = float(bool(row.get('ev_active', False)))
    pv = float(bool(row.get('pv_active', False)))
    weekend = float(bool(row.get('weekend', False)))
    season = str(row.get('season', 'unknown'))
    season_onehot = np.zeros(len(season_vocab), dtype=np.float32)
    if season in season_to_idx:
        season_onehot[season_to_idx[season]] = 1.0
    repeated = np.tile(np.array([hour_norm, non_time_flag, ev, pv, weekend], dtype=np.float32), (N_BUS, 1))
    season_rep = np.tile(season_onehot, (N_BUS, 1))
    node = np.column_stack([
        p / p_scale,
        q / q_scale,
        bus_id_norm,
        slack_flag,
        repeated,
        season_rep,
    ]).astype(np.float32)
    return node

class DNRMetaDataset(Dataset):
    def __init__(self, frame):
        super().__init__()
        self.frame = frame.reset_index(drop=True).copy()
        self.task_labels = [infer_task(row) for _, row in self.frame.iterrows()]
        self.task_to_indices = defaultdict(list)
        for idx, task in enumerate(self.task_labels):
            self.task_to_indices[task].append(idx)

    def len(self):
        return len(self.frame)

    def get(self, idx):
        row = self.frame.iloc[int(idx)]
        x = torch.as_tensor(build_node_features(row), dtype=torch.float32)
        p, q = scenario_load_vectors(row)
        y = torch.zeros(N_LINES, dtype=torch.float32)
        open_lines = parse_open_lines(row['topo'])
        y[torch.as_tensor(open_lines - 1, dtype=torch.long)] = 1.0
        loss_raw = float(row['loss_mw'])
        vmin_raw = float(row['v_min_pu'])
        data = Data(
            x=x,
            edge_index=edge_index.clone(),
            y=y,
            loss_target=torch.tensor([loss_raw], dtype=torch.float32),
            loss_norm_target=torch.tensor([(loss_raw - loss_mu) / loss_sigma], dtype=torch.float32),
            vmin_target=torch.tensor([vmin_raw], dtype=torch.float32),
            vmin_norm_target=torch.tensor([(vmin_raw - vmin_mu) / vmin_sigma], dtype=torch.float32),
            p_mw=torch.as_tensor(p, dtype=torch.float32),
            q_mvar=torch.as_tensor(q, dtype=torch.float32),
            scenario_idx=torch.tensor([idx], dtype=torch.long),
            task_id=torch.tensor([['A_realistic','B_topology_steered','C_stress_sampler'].index(infer_task(row))], dtype=torch.long),
        )
        return data

dataset = DNRMetaDataset(scenarios_df)
print('Dataset size:', len(dataset))
print('Node feature shape:', tuple(dataset[0].x.shape))
print('Task groups:', {k: len(v) for k, v in dataset.task_to_indices.items()})
print('Example target open lines:', torch.where(dataset[0].y > 0.5)[0].add(1).tolist())

## CHAPTER 2: PHYSICS-GATED GRAPH TRANSFORMER ARCHITECTURE

The attention logits are explicitly biased by electrical impedance distance, so long/high-impedance paths are harder to attend across unless the model learns a small gate.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 2.1 StructuralEncoding and impedance distance matrix
# ═══════════════════════════════════════════════════════════════════
def build_adjacency(n_bus, edges):
    A = np.zeros((n_bus, n_bus), dtype=np.float32)
    for i, j in edges.T:
        A[int(i), int(j)] = 1.0
        A[int(j), int(i)] = 1.0
    return A

adj_np = build_adjacency(N_BUS, edge_pairs_np)
deg_np = np.diag(adj_np.sum(axis=1))
laplacian_np = deg_np - adj_np
# Smallest eigenvector is constant; use the next structural_dim components.
eigvals, eigvecs = np.linalg.eigh(laplacian_np)
struct_eig_np = eigvecs[:, 1:structural_dim + 1].astype(np.float32)

# Impedance distance: shortest-path distance with |Z| = sqrt(R^2 + X^2).
G_imp = nx.Graph()
for k, (i, j) in enumerate(edge_pairs_np.T):
    zmag = float(math.sqrt(float(r_ohm[k])**2 + float(x_ohm[k])**2))
    G_imp.add_edge(int(i), int(j), weight=zmag)
Z_dist = np.zeros((N_BUS, N_BUS), dtype=np.float32)
for i in range(N_BUS):
    lengths = nx.single_source_dijkstra_path_length(G_imp, i, weight='weight')
    for j in range(N_BUS):
        Z_dist[i, j] = lengths.get(j, np.max(list(lengths.values())))
Z_dist = Z_dist / max(float(Z_dist.max()), 1e-6)
Z_base_attention = torch.as_tensor(Z_dist, dtype=torch.float32, device=device)

class StructuralEncoding(nn.Module):
    def __init__(self, eig_features):
        super().__init__()
        self.register_buffer('eig_features', torch.as_tensor(eig_features, dtype=torch.float32))

    def forward(self, x, batch):
        # x contains a concatenated mini-batch. Since each graph has the same
        # IEEE 33-bus ordering, repeat the eigenvectors for each graph.
        dense_x, mask = to_dense_batch(x, batch)
        bsz, n_nodes, _ = dense_x.shape
        eig = self.eig_features.to(x.device).unsqueeze(0).expand(bsz, -1, -1)
        dense_aug = torch.cat([dense_x, eig], dim=-1)
        return dense_aug[mask]

print('Structural eigenvectors:', struct_eig_np.shape)
print('Impedance attention matrix:', tuple(Z_base_attention.shape))

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 2.2 Impedance-Gated Attention and MetaPhyGT
# ═══════════════════════════════════════════════════════════════════
class ImpedanceGatedAttention(nn.Module):
    """Dense multi-head attention with a learnable impedance penalty.

    Attention(Q,K) = softmax(QK^T / sqrt(d) - softplus(gamma) * Z_base)
    """
    def __init__(self, dim, heads=4, dropout=0.1):
        super().__init__()
        assert dim % heads == 0
        self.dim = dim
        self.heads = heads
        self.head_dim = dim // heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.proj = nn.Linear(dim, dim)
        self.gamma = nn.Parameter(torch.tensor(0.1))
        self.dropout = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.ff = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(4 * dim, dim))

    def forward(self, x, batch, z_matrix):
        dense_x, mask = to_dense_batch(x, batch)
        B, N, D = dense_x.shape
        qkv = self.qkv(dense_x).view(B, N, 3, self.heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        impedance_penalty = F.softplus(self.gamma) * z_matrix[:N, :N].view(1, 1, N, N)
        scores = scores - impedance_penalty
        key_mask = mask.view(B, 1, 1, N)
        scores = scores.masked_fill(~key_mask, -1e9)
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, N, D)
        h = self.norm1(dense_x + self.proj(out))
        h = self.norm2(h + self.ff(h))
        return h[mask]

class MetaPhyGT(nn.Module):
    def __init__(self, node_dim, hidden_dim=96, heads=4, layers=3, structural_dim=6, dropout=0.1):
        super().__init__()
        self.structural = StructuralEncoding(struct_eig_np[:, :structural_dim])
        self.input = nn.Linear(node_dim + structural_dim, hidden_dim)
        self.layers = nn.ModuleList([ImpedanceGatedAttention(hidden_dim, heads=heads, dropout=dropout) for _ in range(layers)])
        self.edge_decoder = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim, 1)
        )
        self.physics_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim, 2)
        )

    def forward(self, data):
        x, batch = data.x, data.batch
        h = self.structural(x, batch)
        h = self.input(h)
        for layer in self.layers:
            h = layer(h, batch, Z_base_attention)
        dense_h, mask = to_dense_batch(h, batch)
        B, N, H = dense_h.shape
        src = edge_index[0].to(h.device)
        dst = edge_index[1].to(h.device)
        edge_emb = torch.cat([dense_h[:, src, :], dense_h[:, dst, :]], dim=-1)
        edge_logits = self.edge_decoder(edge_emb).squeeze(-1)  # [B, 37], open-line logits
        pooled = (dense_h * mask.unsqueeze(-1)).sum(dim=1) / mask.sum(dim=1, keepdim=True).clamp_min(1)
        physics = self.physics_head(pooled)
        pred_loss_norm = physics[:, 0]
        pred_vmin = 0.75 + 0.35 * torch.sigmoid(physics[:, 1])  # physically plausible range
        return edge_logits, pred_loss_norm, pred_vmin

node_dim = dataset[0].x.shape[1]
model = MetaPhyGT(node_dim=node_dim, hidden_dim=hidden_dim, heads=num_heads, layers=num_layers,
                  structural_dim=structural_dim, dropout=dropout).to(device)
print(model)

## CHAPTER 3: THE REPTILE + JOINT PHYSICS-INFORMED LOSS LOOP

The loss couples topology decisions, loss regression, and voltage safety. Reptile then optimizes for rapid adaptation across realistic, topology-steered, and stress tasks.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 3.1 Physics-informed multi-objective loss
# ═══════════════════════════════════════════════════════════════════
# HDF5-provided loss weights are honored when present. If the file only stores
# diagnostic lambda recommendations, those scale the global physics weights.
hdf_lambda_physics = None
hdf_lambda_voltage = None
if not loss_weights_df.empty:
    numeric = loss_weights_df.select_dtypes(include=[np.number])
    if 'lambda_physics' in numeric.columns:
        hdf_lambda_physics = float(numeric['lambda_physics'].iloc[0])
    if 'lambda_voltage' in numeric.columns:
        hdf_lambda_voltage = float(numeric['lambda_voltage'].iloc[0])

# Long-tailed line-open imbalance: per-line positive class weights.
y_stack = torch.stack([dataset[i].y for i in range(len(dataset))])
pos = y_stack.sum(dim=0)
neg = len(dataset) - pos
edge_pos_weight = (neg / pos.clamp_min(1.0)).clamp(1.0, 50.0).to(device)

w_mse_eff = float(hdf_lambda_physics) if hdf_lambda_physics is not None else w_mse
w_volt_eff = float(hdf_lambda_voltage) if hdf_lambda_voltage is not None else w_volt
print(f'Effective weights: w_bce={w_bce}, w_mse={w_mse_eff:.4f}, w_volt={w_volt_eff:.4f}')

loss_sigma_t = torch.tensor(loss_sigma, dtype=torch.float32, device=device)
loss_mu_t = torch.tensor(loss_mu, dtype=torch.float32, device=device)

def multi_objective_loss(model, batch):
    batch = batch.to(device)
    edge_logits, pred_loss_norm, pred_vmin = model(batch)
    true_edges = batch.y.view(edge_logits.shape[0], N_LINES)
    true_loss_norm = batch.loss_norm_target.view(-1)
    bce = F.binary_cross_entropy_with_logits(edge_logits, true_edges, pos_weight=edge_pos_weight)
    mse = F.mse_loss(pred_loss_norm, true_loss_norm)
    voltage_penalty = torch.relu(V_MIN_PU - pred_vmin).pow(2).mean()
    total = w_bce * bce + w_mse_eff * mse + w_volt_eff * voltage_penalty
    return total, {'bce': float(bce.detach().cpu()), 'mse': float(mse.detach().cpu()), 'volt': float(voltage_penalty.detach().cpu())}

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 3.2 Reptile meta-learning implementation
# ═══════════════════════════════════════════════════════════════════
all_indices = np.arange(len(dataset))
rng = np.random.default_rng(random_seed)
rng.shuffle(all_indices)
val_count = max(1, int(0.20 * len(all_indices)))
val_indices = all_indices[:val_count].tolist()
train_indices = all_indices[val_count:].tolist()

train_by_task = defaultdict(list)
for idx in train_indices:
    train_by_task[dataset.task_labels[idx]].append(idx)
print('Train task sizes:', {k: len(v) for k, v in train_by_task.items()})
print('Validation size:', len(val_indices))

def make_loader(indices, batch_size, shuffle=True):
    return DataLoader([dataset[i] for i in indices], batch_size=batch_size, shuffle=shuffle)

def reptile_step(meta_model, task_indices):
    support = rng.choice(task_indices, size=min(support_size, len(task_indices)), replace=len(task_indices) < support_size)
    fast_model = copy.deepcopy(meta_model).to(device)
    opt = torch.optim.Adam(fast_model.parameters(), lr=inner_lr)
    support_loader = make_loader(support.tolist(), batch_size=len(support), shuffle=True)
    for _ in range(inner_steps):
        for batch in support_loader:
            opt.zero_grad()
            loss, _ = multi_objective_loss(fast_model, batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(fast_model.parameters(), 2.0)
            opt.step()
    with torch.no_grad():
        for p, fp in zip(meta_model.parameters(), fast_model.parameters()):
            p.add_(meta_lr * (fp - p))

def evaluate_loss(meta_model, indices):
    meta_model.eval()
    losses = []
    parts = defaultdict(list)
    with torch.no_grad():
        for batch in make_loader(indices, batch_size=validation_batch_size, shuffle=False):
            loss, logs = multi_objective_loss(meta_model, batch)
            losses.append(float(loss.cpu()))
            for k, v in logs.items():
                parts[k].append(v)
    return float(np.mean(losses)), {k: float(np.mean(v)) for k, v in parts.items()}

history = []
print('Starting Reptile meta-training...')
for epoch in range(1, epochs + 1):
    model.train()
    task_name = random.choice(list(train_by_task.keys()))
    reptile_step(model, train_by_task[task_name])
    if epoch == 1 or epoch % validation_interval == 0 or epoch == epochs:
        train_eval = rng.choice(train_indices, size=min(len(train_indices), validation_batch_size), replace=False).tolist()
        train_loss, train_parts = evaluate_loss(model, train_eval)
        val_loss, val_parts = evaluate_loss(model, val_indices)
        history.append({'epoch': epoch, 'meta_train_loss': train_loss, 'meta_val_loss': val_loss,
                        'task': task_name, **{f'train_{k}': v for k, v in train_parts.items()},
                        **{f'val_{k}': v for k, v in val_parts.items()}})
        print(f'Epoch {epoch:03d}/{epochs} task={task_name:20s} train={train_loss:.4f} val={val_loss:.4f}')

history_df = pd.DataFrame(history)

## CHAPTER 4: PHYSICAL OFFSET ANALYSIS & PANDAPOWER COMPARATIVE CRITIC

The critic projects raw edge probabilities into a radial topology, injects that topology into pandapower, and measures physical offsets.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 4.1 PhysicalValidationCritic
# ═══════════════════════════════════════════════════════════════════
def create_pp_network():
    net = pp.create_empty_network(sn_mva=BASE_MVA)
    for _ in range(N_BUS):
        pp.create_bus(net, vn_kv=VN_KV)
    pp.create_ext_grid(net, bus=0, vm_pu=1.0, va_degree=0.0)
    for bus in range(1, N_BUS):
        pp.create_load(net, bus=bus, p_mw=0.0, q_mvar=0.0)
    for fb, tb, r, x in zip(from_bus, to_bus, r_ohm, x_ohm):
        pp.create_line_from_parameters(net, int(fb), int(tb), 1.0, float(r), float(x), 0.0, 1.0)
    return net

class PhysicalValidationCritic:
    def __init__(self):
        self.net = create_pp_network()
        self.graph_edges = [(int(i), int(j), k) for k, (i, j) in enumerate(edge_pairs_np.T)]

    def radial_projection(self, open_logits):
        # Convert open probabilities to active-edge scores and choose a maximum
        # spanning tree. The complement is exactly 5 open switches.
        open_prob = torch.sigmoid(open_logits.detach()).cpu().numpy()
        G = nx.Graph()
        G.add_nodes_from(range(N_BUS))
        for i, j, k in self.graph_edges:
            G.add_edge(i, j, line_idx=k, weight=float(1.0 - open_prob[k]))
        tree = nx.maximum_spanning_tree(G, weight='weight')
        active = {d['line_idx'] for _, _, d in tree.edges(data=True)}
        open_idx = sorted(set(range(N_LINES)) - active)
        if len(open_idx) != N_OPEN:
            # Fallback: keep the highest-probability opens while preserving size.
            open_idx = np.argsort(-open_prob)[:N_OPEN].tolist()
        return np.array(open_idx, dtype=np.int64)

    def runpp(self, p_mw, q_mvar, open_idx):
        for load_idx, bus in enumerate(range(1, N_BUS)):
            self.net.load.at[load_idx, 'p_mw'] = float(p_mw[bus])
            self.net.load.at[load_idx, 'q_mvar'] = float(q_mvar[bus])
        self.net.line.loc[:, 'in_service'] = True
        self.net.line.loc[open_idx, 'in_service'] = False
        last_error = None
        for algorithm in ['bfsw', 'nr', 'iwamoto_nr']:
            try:
                pp.runpp(self.net, algorithm=algorithm, init='flat', tolerance_mva=1e-9,
                         max_iteration=200, calculate_voltage_angles=False, numba=False)
                if bool(self.net.converged):
                    return {
                        'converged': True,
                        'algorithm': algorithm,
                        'loss_mw': float(self.net.res_line.pl_mw.sum()),
                        'vmin_pu': float(self.net.res_bus.vm_pu.min()),
                        'voltage_violation': bool(self.net.res_bus.vm_pu.min() < V_MIN_PU),
                    }
            except Exception as exc:
                last_error = exc
        return {'converged': False, 'algorithm': None, 'loss_mw': np.nan, 'vmin_pu': np.nan,
                'voltage_violation': True, 'error': str(last_error)}

    def evaluate(self, meta_model, indices):
        meta_model.eval()
        rows = []
        with torch.no_grad():
            for batch in make_loader(indices, batch_size=1, shuffle=False):
                batch = batch.to(device)
                edge_logits, pred_loss_norm, pred_vmin = meta_model(batch)
                open_idx = self.radial_projection(edge_logits[0])
                p_batch = batch.p_mw.view(-1, N_BUS)
                q_batch = batch.q_mvar.view(-1, N_BUS)
                pp_res = self.runpp(p_batch[0].detach().cpu().numpy(),
                                    q_batch[0].detach().cpu().numpy(), open_idx)
                pred_loss = float((pred_loss_norm[0] * loss_sigma_t + loss_mu_t).detach().cpu())
                true_loss = float(batch.loss_target.view(-1)[0].detach().cpu())
                true_vmin = float(batch.vmin_target.view(-1)[0].detach().cpu())
                rows.append({
                    'scenario_idx': int(batch.scenario_idx.view(-1)[0].detach().cpu()),
                    'pred_open_lines': tuple((open_idx + 1).tolist()),
                    'pred_loss_mw': pred_loss,
                    'pred_vmin_pu': float(pred_vmin[0].detach().cpu()),
                    'teacher_loss_mw': true_loss,
                    'teacher_vmin_pu': true_vmin,
                    'pp_loss_mw': pp_res['loss_mw'],
                    'pp_vmin_pu': pp_res['vmin_pu'],
                    'delta_loss': abs(pred_loss - pp_res['loss_mw']) if pp_res['converged'] else np.nan,
                    'delta_voltage_offset': abs(float(pred_vmin[0].detach().cpu()) - pp_res['vmin_pu']) if pp_res['converged'] else np.nan,
                    'pp_voltage_violation': pp_res['voltage_violation'],
                    'pp_converged': pp_res['converged'],
                    'pp_algorithm': pp_res['algorithm'],
                })
        return pd.DataFrame(rows)

critic = PhysicalValidationCritic()
critic_indices = val_indices[:min(critic_cases_per_eval, len(val_indices))]
critic_df = critic.evaluate(model, critic_indices)
display(critic_df.head())
print('Pandapower converged:', critic_df['pp_converged'].mean())
print('Mean ΔLoss:', critic_df['delta_loss'].mean())
print('Mean ΔVoltage:', critic_df['delta_voltage_offset'].mean())

## CHAPTER 5: REAL-TIME CONVERGENCE & ANALYSIS VISUALIZATION

Static plotting suite for convergence, physical offsets, and topology decisions.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 5.1 Visualization suite
# ═══════════════════════════════════════════════════════════════════
def plot_training_and_physics(history_df, critic_df, dataset, model):
    sns.set_theme(style='whitegrid')
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))

    if not history_df.empty:
        axes[0].plot(history_df['epoch'], history_df['meta_train_loss'], marker='o', label='Meta-Train')
        axes[0].plot(history_df['epoch'], history_df['meta_val_loss'], marker='s', label='Meta-Validation')
    axes[0].set_title('Loss Convergence Tracking')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Joint Physics Loss')
    axes[0].legend()

    offsets = critic_df[['delta_loss', 'delta_voltage_offset']].melt(var_name='metric', value_name='offset').dropna()
    if not offsets.empty:
        sns.violinplot(data=offsets, x='metric', y='offset', ax=axes[1], inner='quartile')
    axes[1].set_title('Physical Error Metric Distribution')
    axes[1].set_xlabel('Offset metric')
    axes[1].set_ylabel('Absolute offset')

    tie_idx = np.array([32, 33, 34, 35, 36], dtype=np.int64)
    eval_indices = val_indices[:min(128, len(val_indices))]
    if eval_indices:
        probs, targets = [], []
        model.eval()
        with torch.no_grad():
            for batch in make_loader(eval_indices, batch_size=validation_batch_size, shuffle=False):
                batch = batch.to(device)
                edge_logits, _, _ = model(batch)
                probs.append(torch.sigmoid(edge_logits[:, tie_idx]).cpu().numpy())
                targets.append(batch.y.view(edge_logits.shape[0], N_LINES)[:, tie_idx].cpu().numpy())
        prob_mat = np.vstack(probs)
        target_mat = np.vstack(targets)
        heat = np.concatenate([target_mat, prob_mat], axis=1)
        sns.heatmap(heat, ax=axes[2], cmap='viridis', cbar=True)
        axes[2].axvline(5, color='white', linewidth=2)
        axes[2].set_xticks(np.arange(10) + 0.5)
        axes[2].set_xticklabels([f'T{l}' for l in range(33, 38)] + [f'P{l}' for l in range(33, 38)], rotation=45)
    axes[2].set_title('Tie-Switch Targets vs Predicted Opening Probability')
    axes[2].set_xlabel('Exact target | predicted probability')
    axes[2].set_ylabel('Validation scenario')

    plt.tight_layout()
    plt.show()

plot_training_and_physics(history_df, critic_df, dataset, model)

## CHAPTER 6: RUN EXECUTION & HYPERPARAMETER TUNING

The notebook above is executable top-to-bottom. The orchestrator below provides a clean function interface for repeated experiments after tuning globals in the first cell.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 6.1 Main orchestrator for repeated runs
# ═══════════════════════════════════════════════════════════════════
def main():
    print('═' * 80)
    print('Meta-PhyGT DNR experiment complete')
    print('═' * 80)
    print(f'Dataset scenarios      : {len(dataset):,}')
    print(f'Tasks                  : { {k: len(v) for k, v in dataset.task_to_indices.items()} }')
    print(f'Hyperparameters        : inner_lr={inner_lr}, meta_lr={meta_lr}, inner_steps={inner_steps}, epochs={epochs}')
    if not history_df.empty:
        print(f'Final meta-train loss  : {history_df.meta_train_loss.iloc[-1]:.5f}')
        print(f'Final meta-val loss    : {history_df.meta_val_loss.iloc[-1]:.5f}')
    if not critic_df.empty:
        print(f'Critic cases           : {len(critic_df)}')
        print(f'Pandapower convergence : {critic_df.pp_converged.mean()*100:.1f}%')
        print(f'Mean ΔLoss             : {critic_df.delta_loss.mean():.6f} MW')
        print(f'Mean ΔVoltage Offset   : {critic_df.delta_voltage_offset.mean():.6f} pu')
        print(f'Voltage violations     : {critic_df.pp_voltage_violation.sum()}')
    print('═' * 80)
    return {
        'history': history_df,
        'critic': critic_df,
        'model': model,
        'dataset': dataset,
    }

results = main()